In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
deposits_size = pd.read_csv('../int/deposits_size.csv')
deposits_category = pd.read_csv('../int/deposits_category.csv')
deposits_pool = pd.read_csv('../int/deposits_pool.csv')
staked_pool_size = pd.read_csv('../int/active_validators_size.csv')
staked_category = pd.read_csv('../int/active_validators_category.csv')
staked_pool = pd.read_csv('../int/active_validators_pool.csv')

In [3]:
staked_pool_size.loc[:, staked_pool_size.columns != 'slot'] *= 32
staked_category.loc[:, staked_category.columns != 'slot'] *= 32
staked_pool.loc[:, staked_pool.columns != 'slot'] *= 32

In [4]:
eth_price = pd.read_csv('../int/eth_price.csv')
eth_price = eth_price[eth_price['slot'] >= 6206400]
eth_price

,slot,price,price_pct_change
6174901,6206400,1892.938911,-0.008871
6174902,6206401,1892.938911,-0.008871
6174903,6206402,1892.938911,-0.008871
6174904,6206403,1892.938911,-0.008871
6174905,6206404,1892.938911,-0.008871
...,...,...,...
8954672,8986171,2976.090208,-0.014066
8954673,8986172,2976.090208,-0.014066
8954674,8986173,2976.090208,-0.014066
8954675,8986174,2976.090208,-0.014066


In [5]:
deposits_size_set = deposits_size
deposits_size_set['slot'] = deposits_size_set['slot'] // 300 * 300

# Group by the day and get the last slot of each day and sum values for all columns
deposits_size_set = deposits_size_set.groupby('slot').agg(
    {
        'slot': 'last',
        '1': 'sum',
        '2-5': 'sum',
        '6-19': 'sum',
        '20-99': 'sum',
        '100+': 'sum',
        'total': 'sum'
    }
).reset_index(drop=True)

deposits_size_set


,slot,1,2-5,6-19,20-99,100+,total
0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,300,0.0,0.0,0.0,0.0,0.0,0.0
2,600,0.0,0.0,0.0,0.0,0.0,0.0
3,900,0.0,0.0,0.0,0.0,0.0,0.0
4,1200,6912.0,7840.0,9216.0,17440.0,83520.0,124928.0
...,...,...,...,...,...,...,...
29949,8984700,0.0,0.0,0.0,0.0,0.0,0.0
29950,8985000,0.0,0.0,0.0,0.0,0.0,0.0
29951,8985300,0.0,0.0,0.0,0.0,0.0,0.0
29952,8985600,192.0,0.0,0.0,256.0,6878.0,7326.0


In [6]:
deposits_category_set = deposits_category
deposits_category_set['slot'] = deposits_category_set['slot'] // 300 * 300

# Create a dictionary for aggregation
agg_dict = {col: 'sum' for col in deposits_category_set.columns if col != 'slot'}
agg_dict['slot'] = 'last'

# Group by 'slot' and apply the aggregation
deposits_category_set = deposits_category_set.groupby('slot').agg(agg_dict).reset_index(drop=True)

deposits_category_set

,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total,slot
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,300
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,600
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,900
4,38368.0,0.0,3456.0,9248.0,1984.0,71872.0,124928.0,1200
...,...,...,...,...,...,...,...,...
29949,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8984700
29950,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8985000
29951,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8985300
29952,800.0,160.0,94.0,0.0,1824.0,4448.0,7326.0,8985600


In [7]:
deposits_category_set

,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total,slot
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,300
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,600
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,900
4,38368.0,0.0,3456.0,9248.0,1984.0,71872.0,124928.0,1200
...,...,...,...,...,...,...,...,...
29949,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8984700
29950,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8985000
29951,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8985300
29952,800.0,160.0,94.0,0.0,1824.0,4448.0,7326.0,8985600


In [8]:
deposits_pool_set = deposits_pool
deposits_pool_set['slot'] = deposits_pool_set['slot'] // 300 * 300

# Create a dictionary for aggregation
agg_dict = {col: 'sum' for col in deposits_pool_set.columns if col != 'slot'}
agg_dict['slot'] = 'last'

# Group by 'slot' and apply the aggregation
deposits_pool_set = deposits_pool_set.groupby('slot').agg(agg_dict).reset_index(drop=True)

deposits_pool_set

,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total,slot
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,300
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,600
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,900
4,0.0,3936.0,0.0,0.0,0.0,64.0,0.0,0.0,0.0,120928.0,0.0,124928.0,1200
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29949,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8984700
29950,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8985000
29951,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8985300
29952,0.0,0.0,0.0,0.0,288.0,1824.0,0.0,0.0,0.0,5120.0,94.0,7326.0,8985600


In [9]:
staked_pool_size_set = staked_pool_size[staked_pool_size['slot'].isin(deposits_size_set['slot'])]
staked_pool_size_set.reset_index(inplace=True)
staked_pool_size_set = staked_pool_size_set.drop(columns=['index'])
staked_pool_size_set

,slot,1,100+,2-5,20-99,6-19,total
0,0.0,32640.0,435168.0,33568.0,114176.0,58464.0,674016.0
1,300.0,32640.0,435168.0,33568.0,114176.0,58464.0,674016.0
2,600.0,32640.0,435168.0,33568.0,114176.0,58464.0,674016.0
3,900.0,32640.0,435168.0,33568.0,114176.0,58464.0,674016.0
4,1200.0,32640.0,435168.0,33568.0,114176.0,58464.0,674016.0
...,...,...,...,...,...,...,...
29949,8984700.0,315072.0,29434144.0,317600.0,1419456.0,578624.0,32064896.0
29950,8985000.0,314976.0,29434048.0,317600.0,1419456.0,578624.0,32064704.0
29951,8985300.0,315040.0,29435904.0,317600.0,1419456.0,578624.0,32066624.0
29952,8985600.0,315040.0,29438272.0,317600.0,1419456.0,578624.0,32068992.0


In [10]:
staked_category_set = staked_category[staked_category['slot'].isin(deposits_category_set['slot'])]
staked_category_set.reset_index(inplace=True)
staked_category_set = staked_category_set.drop(columns=['index'])
staked_category_set

,slot,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total
0,0.0,110976.0,0.0,17888.0,73888.0,19904.0,451360.0,674016.0
1,300.0,110976.0,0.0,17888.0,73888.0,19904.0,451360.0,674016.0
2,600.0,110976.0,0.0,17888.0,73888.0,19904.0,451360.0,674016.0
3,900.0,110976.0,0.0,17888.0,73888.0,19904.0,451360.0,674016.0
4,1200.0,110976.0,0.0,17888.0,73888.0,19904.0,451360.0,674016.0
...,...,...,...,...,...,...,...,...
29949,8984700.0,8189440.0,2489376.0,10611232.0,538784.0,1806592.0,8429472.0,32064896.0
29950,8985000.0,8189056.0,2491936.0,10611104.0,538784.0,1806528.0,8427296.0,32064704.0
29951,8985300.0,8189056.0,2493568.0,10611040.0,538784.0,1806240.0,8427936.0,32066624.0
29952,8985600.0,8189056.0,2493568.0,10610912.0,538784.0,1806400.0,8430272.0,32068992.0


In [11]:
staked_pool_set = staked_pool[staked_pool['slot'].isin(deposits_pool_set['slot'])]
staked_pool_set.reset_index(inplace=True)
staked_pool_set = staked_pool_set.drop(columns=['index'])
staked_pool_set

,slot,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total
0,0.0,32.0,92032.0,0.0,0.0,0.0,416.0,0.0,0.0,0.0,580896.0,640.0,674016.0
1,300.0,32.0,92032.0,0.0,0.0,0.0,416.0,0.0,0.0,0.0,580896.0,640.0,674016.0
2,600.0,32.0,92032.0,0.0,0.0,0.0,416.0,0.0,0.0,0.0,580896.0,640.0,674016.0
3,900.0,32.0,92032.0,0.0,0.0,0.0,416.0,0.0,0.0,0.0,580896.0,640.0,674016.0
4,1200.0,32.0,92032.0,0.0,0.0,0.0,416.0,0.0,0.0,0.0,580896.0,640.0,674016.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29949,8984700.0,1125344.0,608416.0,4427168.0,1077280.0,759936.0,471136.0,9302112.0,479232.0,348064.0,12674784.0,791424.0,32064896.0
29950,8985000.0,1125344.0,608416.0,4426880.0,1079840.0,759936.0,471136.0,9302112.0,479168.0,348064.0,12672448.0,791360.0,32064704.0
29951,8985300.0,1125344.0,608416.0,4426880.0,1081440.0,759936.0,471136.0,9302112.0,479168.0,348064.0,12672832.0,791296.0,32066624.0
29952,8985600.0,1125344.0,608416.0,4426880.0,1081440.0,759936.0,471296.0,9302112.0,479168.0,348064.0,12675168.0,791168.0,32068992.0


In [12]:
deposits_size_set.set_index('slot', inplace=True)
staked_pool_size_set.set_index('slot', inplace=True)
deposits_percentage_size = deposits_size_set.divide(staked_pool_size_set, fill_value=0)
deposits_percentage_size.reset_index(inplace=True)
deposits_percentage_size

,slot,1,100+,2-5,20-99,6-19,total
0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,300,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,600,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,900,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,1200,0.211765,0.191926,0.233556,0.152747,0.157635,0.185349
...,...,...,...,...,...,...,...
29949,8984700,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
29950,8985000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
29951,8985300,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
29952,8985600,0.000609,0.000234,0.000000,0.000180,0.000000,0.000228


In [13]:
deposits_category_set.set_index('slot', inplace=True)
staked_category_set.set_index('slot', inplace=True)
deposits_percentage_category = deposits_category_set.divide(staked_category_set, fill_value=0)
deposits_percentage_category.reset_index(inplace=True)
deposits_percentage_category

,slot,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total
0,0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
1,300,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
2,600,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
3,900,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
4,1200,0.345732,NaN,0.193202,0.125162,0.099678,0.159234,0.185349
...,...,...,...,...,...,...,...,...
29949,8984700,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
29950,8985000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
29951,8985300,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
29952,8985600,0.000098,0.000064,0.000009,0.000000,0.001010,0.000528,0.000228


In [14]:
staked_category_set['Liquid Restaking'].to_csv('wtf.csv')

In [15]:
deposits_percentage_category[['slot','Liquid Restaking']].to_csv('wtf.csv')

In [16]:
deposits_pool_set.set_index('slot', inplace=True)
staked_pool_set.set_index('slot', inplace=True)
deposits_percentage_pool = deposits_pool_set.divide(staked_pool_set, fill_value=0)
deposits_percentage_pool.reset_index(inplace=True)
deposits_percentage_pool

,slot,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total
0,0,0.0,0.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000
1,300,0.0,0.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000
2,600,0.0,0.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000
3,900,0.0,0.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000
4,1200,0.0,0.042768,NaN,NaN,NaN,0.153846,NaN,NaN,NaN,0.208175,0.000000,0.185349
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29949,8984700,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000
29950,8985000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000
29951,8985300,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000
29952,8985600,0.0,0.000000,0.0,0.0,0.000379,0.003870,0.0,0.0,0.0,0.000404,0.000119,0.000228


In [17]:
columns = ['1', '100+', '2-5', '20-99', '6-19', 'total']
for col in columns:
    print(deposits_percentage_size[col].mean())

0.0001036876516686348
0.00018007505088583426
9.995230738237577e-05
0.00010234118216427566
9.78244262338285e-05
0.0001628941541956807


In [18]:
from scipy import stats
import numpy as np
import pandas as pd

# Merge the two DataFrames on the slot column
price_elasticity_size = pd.merge(deposits_percentage_size, eth_price[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_size.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_size['elasticity_total'] = price_elasticity_size['total'] / price_elasticity_size['price_pct_change']
price_elasticity_size['elasticity_total'] = price_elasticity_size['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_size['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = ['1', '2-5', '6-19', '20-99', '100+']
for col in columns:
    price_elasticity_size[f'elasticity_{col}'] = price_elasticity_size[col] / price_elasticity_size['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_size[f'elasticity_{col}'] = price_elasticity_size[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_size[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_size)


Elasticity Analysis Results:
total: Mean Elasticity = 5.5356274111829226e-05, t(Mean) = -, SD = 0.18587738123470618, N = 9266, p-value = -
1: Mean Elasticity = 0.0005406465663029188, t(Mean) = 0.17613058731534423, SD = 0.18919148285916262, N = 9266, p-value = 0.860193276664147
2-5: Mean Elasticity = 0.002150923707090075, t(Mean) = 0.7137352353507791, SD = 0.21290025277573932, N = 9266, p-value = 0.47540006695138715
6-19: Mean Elasticity = 0.0011851954189421568, t(Mean) = 0.3219061286921563, SD = 0.28212987490891317, N = 9266, p-value = 0.7475280068126164
20-99: Mean Elasticity = -0.001464573998193043, t(Mean) = -0.5056799467316442, SD = 0.22172434424232118, N = 9266, p-value = 0.6130875463665088
100+: Mean Elasticity = 6.846605135860485e-05, t(Mean) = 0.0047065418528787085, SD = 0.19323910500080166, N = 9266, p-value = 0.9962447875263674
         slot         1      100+  2-5     20-99  6-19     total  \
0     6206400  0.000301  0.000313  0.0  0.000562   0.0  0.000313   
1     6206700 

In [19]:
from scipy import stats
import numpy as np
import pandas as pd

# Merge the two DataFrames on the slot column
price_elasticity_category = pd.merge(deposits_percentage_category, eth_price[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_category.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_category['elasticity_total'] = price_elasticity_category['total'] / price_elasticity_category['price_pct_change']
price_elasticity_category['elasticity_total'] = price_elasticity_category['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_category['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = deposits_percentage_category.columns.drop('slot')
for col in columns:
    price_elasticity_category[f'elasticity_{col}'] = price_elasticity_category[col] / price_elasticity_category['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_category[f'elasticity_{col}'] = price_elasticity_category[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_category[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_category)

Elasticity Analysis Results:
total: Mean Elasticity = 5.5356274111829226e-05, t(Mean) = 0.0, SD = 0.18587738123470618, N = 9266, p-value = 1.0
CEX: Mean Elasticity = 0.0022962343455621283, t(Mean) = 0.6860402931762059, SD = 0.25359736609394046, N = 9266, p-value = 0.4926970612707401
Liquid Restaking: Mean Elasticity = -0.0802123197743997, t(Mean) = -0.7852654138566901, SD = 9.837683805846478, N = 9266, p-value = 0.4323180347632979
Liquid Staking: Mean Elasticity = -0.0021346890973383084, t(Mean) = -0.7609492738088341, SD = 0.20542916204065187, N = 9266, p-value = 0.4466971453617026
Solo Stakers: Mean Elasticity = -0.0003010954089901852, t(Mean) = -0.114035555724788, SD = 0.23660913463879155, N = 9266, p-value = 0.9092109125908427
Staking Pools: Mean Elasticity = -0.0005901430305478265, t(Mean) = -0.13857388554267236, SD = 0.40805326015899596, N = 9266, p-value = 0.8897890349419326
Unidentified: Mean Elasticity = 0.001107638738984103, t(Mean) = 0.2727261947278241, SD = 0.321548968933861

In [20]:
price_elasticity_category.columns

Index(['slot', 'CEX', 'Liquid Restaking', 'Liquid Staking', 'Solo Stakers',
       'Staking Pools', 'Unidentified', 'total', 'price_pct_change',
       'elasticity_total', 'elasticity_CEX', 'elasticity_Liquid Restaking',
       'elasticity_Liquid Staking', 'elasticity_Solo Stakers',
       'elasticity_Staking Pools', 'elasticity_Unidentified'],
      dtype='object')

In [21]:
from scipy import stats
import numpy as np
import pandas as pd

# Merge the two DataFrames on the slot column
price_elasticity_pool = pd.merge(deposits_percentage_pool, eth_price[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_pool.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_pool['elasticity_total'] = price_elasticity_pool['total'] / price_elasticity_pool['price_pct_change']
price_elasticity_pool['elasticity_total'] = price_elasticity_pool['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_pool['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = ['Lido', 'Coinbase', 'Binance', 'Rocketpool', 'Kraken', 'OKX', 'Bitcoin Suisse', 'Ledger Live', 'Ether.Fi', 'Mantle', 'Other Stakers']
for col in columns:
    price_elasticity_pool[f'elasticity_{col}'] = price_elasticity_pool[col] / price_elasticity_pool['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_pool[f'elasticity_{col}'] = price_elasticity_pool[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_pool[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_pool)

Elasticity Analysis Results:
total: Mean Elasticity = 5.5356274111829226e-05, t(Mean) = -, SD = 0.18587738123470618, N = 9266, p-value = -
Lido: Mean Elasticity = -0.003927173637483585, t(Mean) = -1.4033548939479645, SD = 0.2001825558795501, N = 9266, p-value = 0.1605278847857476
Coinbase: Mean Elasticity = 0.004508100389685232, t(Mean) = 1.0225107728914542, SD = 0.37572047337089254, N = 9266, p-value = 0.30655747933069516
Binance: Mean Elasticity = -0.0011229357624995666, t(Mean) = -0.17767913807172628, SD = 0.6106942213602676, N = 9266, p-value = 0.8589782513913936
Rocketpool: Mean Elasticity = -6.306998891920472e-05, t(Mean) = -0.038137280991354966, SD = 0.2340906531623749, N = 9266, p-value = 0.9695786589200652
Kraken: Mean Elasticity = 0.004716070380837598, t(Mean) = 0.9324723716604019, SD = 0.44377459437801625, N = 9266, p-value = 0.3511105861965106
OKX: Mean Elasticity = 0.006231882177646863, t(Mean) = 0.8487671474725659, SD = 0.6753781494429837, N = 9266, p-value = 0.3960299320